# ACOS Extract-Classify-ACOS — Kaggle runner

Supports **English** (`rest16`, `laptop`) and **Amharic** (`amharic`) domains.

**Pipeline overview:**
1. Install dependencies
2. Locate and copy the repository
3. Patch known source-file bugs
4. *(Amharic only)* Upload your data and generate tokenized input files
5. Run Step 1 — aspect/opinion span extraction (BertForQuadABSA + CRF)
6. Generate candidate pairs from Step 1 predictions
7. Run Step 2 — category + sentiment classification

**Instructions:** Upload the project as a Kaggle dataset, enable Internet + GPU, set `DOMAIN` in the settings cell, then run all cells in order.

## 1. Install dependencies

In [ ]:
!pip install --upgrade pip --quiet
!pip install torchcrf boto3 --quiet

In [ ]:
# Check GPU and Python environment
import os, sys, subprocess
print('Python:', sys.version)
try:
    print(subprocess.check_output(['nvidia-smi', '-L']).decode()[:500])
except Exception:
    print('nvidia-smi not available or no GPU')
import torch
print('torch.cuda.is_available():', torch.cuda.is_available())

## 2. Locate and copy the repository

In [ ]:
import os, shutil

# Primary path — adjust to match your Kaggle dataset slug
repo_src = '/kaggle/input/datasets/azariyasmekonen/acos-2021/ACOS'
if not os.path.exists(repo_src):
    for name in ['ACOS', 'acos', 'ACOS-main', 'Extract-Classify-ACOS']:
        p = os.path.join('/kaggle/input', name)
        if os.path.exists(p):
            repo_src = p
            break
    else:
        repo_src = None

if repo_src is None and os.path.exists('Extract-Classify-ACOS'):
    repo_src = os.path.abspath('Extract-Classify-ACOS')

if repo_src is None:
    raise RuntimeError(
        'Repository not found. Mount the Kaggle dataset at '
        '/kaggle/input/datasets/azariyasmekonen/acos-2021/ACOS or '
        'place Extract-Classify-ACOS/ in the working directory.'
    )

print('Found repository at', repo_src)
repo_dst = '/kaggle/working/Extract-Classify-ACOS'
if os.path.exists(repo_dst):
    shutil.rmtree(repo_dst)
shutil.copytree(repo_src, repo_dst)
print('Copied to', repo_dst)
os.chdir(repo_dst)
print('CWD:', os.getcwd())

## 3. Patch known source-file bugs

In [ ]:
def patch_file(path, old, new, description):
    with open(path, 'r', encoding='utf-8') as fh:
        src = fh.read()
    if old not in src:
        print(f'[SKIP] {description} — already patched')
        return
    with open(path, 'w', encoding='utf-8') as fh:
        fh.write(src.replace(old, new, 1))
    print(f'[OK]   {description}')

cwd = os.getcwd()

# Bug 1 — dataset_utils.py: hard-coded NFS import path
patch_file(
    os.path.join(cwd, 'dataset_utils.py'),
    "sys.path.insert(0, '/mnt/nfs-storage-titan/BERT/pytorch_pretrained_BERT')\n"
    "from pytorch_pretrained_bert.tokenization import BertTokenizer",
    "from bert_utils.tokenization import BertTokenizer",
    'dataset_utils.py — hard-coded NFS import'
)

# Bug 2 — run_step1.py: undefined ae_loss in gradient accumulation block
patch_file(
    os.path.join(cwd, 'run_step1.py'),
    "                if args.gradient_accumulation_steps > 1:\n"
    "                    loss = loss / args.gradient_accumulation_steps\n"
    "                    ae_loss = ae_loss / args.gradient_accumulation_steps",
    "                if args.gradient_accumulation_steps > 1:\n"
    "                    loss = loss / args.gradient_accumulation_steps",
    'run_step1.py — undefined ae_loss in gradient_accumulation block'
)
patch_file(
    os.path.join(cwd, 'run_step1.py'),
    "                    optimizer.backward(ae_loss)",
    "                    optimizer.backward(loss)",
    'run_step1.py — fp16 backward used ae_loss instead of loss'
)

# Bug 3 — run_step2.py: convert_examples_to_features2nd called with task_name
#          but the function signature expects output_mode
for old_call, desc in [
    (
        "eval_features = convert_examples_to_features2nd(\n"
        "            eval_examples, label_list, args.max_seq_length, tokenizer, task_name)",
        'run_step2.py — eval features: task_name -> output_mode'
    ),
    (
        "train_features = convert_examples_to_features2nd(\n"
        "        train_examples, label_list, args.max_seq_length, tokenizer, task_name)",
        'run_step2.py — train features: task_name -> output_mode'
    ),
    (
        "valid_features = convert_examples_to_features2nd(\n"
        "        valid_examples, label_list, args.max_seq_length, tokenizer, task_name)",
        'run_step2.py — valid features: task_name -> output_mode'
    ),
]:
    patch_file(os.path.join(cwd, 'run_step2.py'),
               old_call,
               old_call.replace(', task_name)', ', output_mode="classification")'),
               desc)

print('\nAll patches done.')

## 4. Settings

Set `DOMAIN` to one of:
- `'rest16'` — English restaurant reviews (SemEval-2016)
- `'laptop'` — English laptop reviews (SemEval-2016)
- `'amharic'` — Amharic reviews (your own annotated data required; sample files are included)

When `DOMAIN = 'amharic'`:
- `BERT_MODEL` is automatically set to `bert-base-multilingual-cased` (mBERT), which supports Ethiopic script.
- `DO_LOWER_CASE` is set to `False` — Amharic uses Ethiopic/Ge'ez script, lowercasing is meaningless.
- You must run the **data preparation cell** (Section 4a) before training.

In [ ]:
# ── User-editable settings ─────────────────────────────────────────────────
DOMAIN = 'amharic'   # 'rest16' | 'laptop' | 'amharic'

EPOCHS_STEP1           = 10
EPOCHS_STEP2           = 10
TRAIN_BATCH_SIZE_STEP1 = 8
TRAIN_BATCH_SIZE_STEP2 = 8
OUTPUT_BASE            = '/kaggle/working'
MAX_SEQ_LENGTH         = 128
LEARNING_RATE_STEP1    = 2e-5
LEARNING_RATE_STEP2    = 5e-5

# ── Domain-specific automatic overrides ───────────────────────────────────
if DOMAIN == 'amharic':
    # mBERT is the best publicly available model covering Ethiopic script.
    # If you have a fine-tuned Amharic BERT (e.g., AfriBERTa), set its path here.
    BERT_MODEL    = 'bert-base-multilingual-cased'
    DO_LOWER_CASE = False   # Ethiopic script has no case distinction
else:
    BERT_MODEL    = 'bert-base-uncased'
    DO_LOWER_CASE = True

lower_flag = '--do_lower_case' if DO_LOWER_CASE else ''
print(f'Domain      : {DOMAIN}')
print(f'BERT model  : {BERT_MODEL}')
print(f'Lower case  : {DO_LOWER_CASE}')

## 4a. Amharic data preparation *(skip if DOMAIN != 'amharic')*

The pipeline expects pre-processed TSV files under `tokenized_data/`.
For English these are already committed to the repo.
For Amharic you need to run the preparation script once after uploading your data.

### What you need to provide

Upload your Amharic quad-annotated TSV files as a Kaggle dataset and mount them (or copy them) so they are accessible. The expected files are:

```
data/Amharic-ACOS/amharic_quad_train.tsv
data/Amharic-ACOS/amharic_quad_dev.tsv
data/Amharic-ACOS/amharic_quad_test.tsv
```

### Annotation format

Each line: `review_text<TAB>asp_start,asp_end CATEGORY#SENTIMENT opi_start,opi_end [<TAB> more quads]`

- Spans are **0-indexed whitespace-tokenised word positions** (same as the English data).
- Use `-1,-1` for implicit aspect or opinion (no explicit span).
- Sentiment: `0`=negative, `1`=neutral, `2`=positive.
- Categories must match the list in `run_classifier_dataset_utils.py` under `domain_type == 'amharic'`.

Sample files with illustrative sentences are included in the repo under `data/Amharic-ACOS/`.
Replace them with your real annotated data before training.

In [ ]:
import os, shutil, subprocess

if DOMAIN == 'amharic':
    cwd = os.getcwd()  # /kaggle/working/Extract-Classify-ACOS

    # ── 1. Locate Amharic data ─────────────────────────────────────────────
    # Try the path where the repo dataset would include the data folder:
    amharic_data_candidates = [
        os.path.join(cwd, '..', 'data', 'Amharic-ACOS'),          # sibling of Extract-Classify-ACOS
        '/kaggle/input/amharic-acos',                              # standalone Kaggle dataset
        '/kaggle/input/datasets/azariyasmekonen/acos-2021/ACOS/data/Amharic-ACOS',
    ]
    amharic_data_dir = None
    for p in amharic_data_candidates:
        p = os.path.normpath(p)
        if os.path.isdir(p):
            amharic_data_dir = p
            break

    if amharic_data_dir is None:
        raise RuntimeError(
            'Amharic data directory not found. '
            'Upload your annotated TSVs (amharic_quad_{train,dev,test}.tsv) '
            'and mount the dataset so it appears at one of:\n  ' +
            '\n  '.join(amharic_data_candidates)
        )
    print('Amharic data found at:', amharic_data_dir)

    # ── 2. Copy data into working repo so the prep script can write outputs ─
    working_data_dir = os.path.join(cwd, 'amharic_data')
    if os.path.exists(working_data_dir):
        shutil.rmtree(working_data_dir)
    shutil.copytree(amharic_data_dir, working_data_dir)
    print('Copied data to:', working_data_dir)

    # ── 3. Add amharic label patch to run_classifier_dataset_utils.py ──────
    # The patch adds the 'amharic' branch to CategorySentiProcessor.get_labels().
    # This is already present in the committed version; the patch_file() call
    # below is a safety net in case an older version of the file was copied.
    utils_path = os.path.join(cwd, 'run_classifier_dataset_utils.py')
    with open(utils_path, 'r', encoding='utf-8') as fh:
        utils_src = fh.read()

    amharic_labels_block = '''
        elif domain_type == 'amharic':
            l = [
                'FOOD#QUALITY', 'FOOD#PRICE', 'FOOD#STYLE_OPTIONS', 'FOOD#GENERAL',
                'DRINKS#QUALITY', 'DRINKS#PRICE', 'DRINKS#STYLE_OPTIONS', 'DRINKS#GENERAL',
                'SERVICE#GENERAL', 'SERVICE#QUALITY',
                'AMBIENCE#GENERAL', 'AMBIENCE#DESIGN_FEATURES',
                'LOCATION#GENERAL', 'LOCATION#ACCESSIBILITY',
                'RESTAURANT#GENERAL', 'RESTAURANT#PRICES', 'RESTAURANT#MISCELLANEOUS',
                'SHOP#GENERAL', 'SHOP#PRICES', 'SHOP#QUALITY', 'SHOP#STYLE_OPTIONS',
                'PRODUCT#GENERAL', 'PRODUCT#QUALITY', 'PRODUCT#PRICE', 'PRODUCT#DESIGN_FEATURES',
                'DELIVERY#GENERAL', 'DELIVERY#QUALITY', 'DELIVERY#PRICE',
                'STAFF#GENERAL', 'STAFF#QUALITY',
                'VALUE#GENERAL',
            ]
        elif domain_type == 'laptop':
'''
    if "domain_type == 'amharic'" not in utils_src:
        # Insert before the laptop branch
        utils_src = utils_src.replace(
            "        elif domain_type == 'laptop':",
            amharic_labels_block
        )
        with open(utils_path, 'w', encoding='utf-8') as fh:
            fh.write(utils_src)
        print('[OK]   run_classifier_dataset_utils.py — added amharic labels')
    else:
        print('[SKIP] run_classifier_dataset_utils.py — amharic labels already present')

    # ── 4. Run prepare_amharic.py to generate tokenized_data/ files ────────
    cmd = (
        f'python tokenized_data/prepare_amharic.py '
        f'--data_dir {working_data_dir} '
        f'--out_dir tokenized_data '
        f'--bert_model {BERT_MODEL}'
    )
    print('\nRunning data prep:\n', cmd)
    proc = subprocess.run(cmd, shell=True, check=False)
    if proc.returncode != 0:
        raise RuntimeError(f'prepare_amharic.py failed with exit code {proc.returncode}')

    # ── 5. Verify expected output files exist ──────────────────────────────
    expected = [
        'tokenized_data/amharic_train_quad_bert.tsv',
        'tokenized_data/amharic_dev_quad_bert.tsv',
        'tokenized_data/amharic_test_quad_bert.tsv',
        'tokenized_data/amharic_train_pair.tsv',
        'tokenized_data/amharic_dev_pair.tsv',
    ]
    missing = [f for f in expected if not os.path.exists(f)]
    if missing:
        raise RuntimeError('Missing tokenized data files:\n  ' + '\n  '.join(missing))
    print('\nAll tokenized data files present. Ready to train.')

else:
    print(f'DOMAIN={DOMAIN}: skipping Amharic data preparation.')

## 5. Step 1 — Aspect & Opinion Span Extraction

In [ ]:
import subprocess, os

STEP1_DIR_NAME = DOMAIN + '_1st'
step1_out = os.path.join(OUTPUT_BASE, 'output', 'Extract-Classify-QUAD', STEP1_DIR_NAME)

cmd = (
    f'python run_step1.py --task_name quad --do_train --do_eval '
    f'--domain_type {DOMAIN} --model_type quad {lower_flag} '
    f'--data_dir . --bert_model {BERT_MODEL} '
    f'--max_seq_length {MAX_SEQ_LENGTH} '
    f'--train_batch_size {TRAIN_BATCH_SIZE_STEP1} '
    f'--learning_rate {LEARNING_RATE_STEP1} '
    f'--num_train_epochs {EPOCHS_STEP1} '
    f'--output_dir {step1_out}'
)
print('Running Step 1:\n', cmd)
proc = subprocess.run(cmd, shell=True, check=False)
print('Step 1 exit code:', proc.returncode)

## 6. Generate Candidate Pairs from Step 1 Predictions

In [ ]:
# get_1st_pairs.py reads <base_dir>/output/Extract-Classify-QUAD/<domain>_1st/pred4pipeline.txt
# and writes tokenized_data/<domain>_test_pair_1st.tsv
cmd = f'python tokenized_data/get_1st_pairs.py . {DOMAIN}'
print('Running:', cmd)
proc = subprocess.run(cmd, shell=True, check=False)
print('get_1st_pairs exit code:', proc.returncode)

pair_file = os.path.join('tokenized_data', DOMAIN + '_test_pair_1st.tsv')
if os.path.exists(pair_file):
    with open(pair_file, encoding='utf-8') as f:
        lines = f.readlines()
    print(f'Generated {len(lines)} candidate pairs -> {pair_file}')
else:
    print(f'WARNING: {pair_file} not found — check Step 1 output')

## 7. Step 2 — Category & Sentiment Classification

In [ ]:
step2_out = os.path.join(OUTPUT_BASE, 'output', 'Extract-Classify-QUAD', DOMAIN + '_2nd')

cmd = (
    f'python run_step2.py --task_name categorysenti --do_train --do_eval '
    f'--domain_type {DOMAIN} --model_type categorysenti {lower_flag} '
    f'--data_dir . --bert_model {BERT_MODEL} '
    f'--max_seq_length {MAX_SEQ_LENGTH} '
    f'--train_batch_size {TRAIN_BATCH_SIZE_STEP2} '
    f'--learning_rate {LEARNING_RATE_STEP2} '
    f'--num_train_epochs {EPOCHS_STEP2} '
    f'--output_dir {step2_out}'
)
print('Running Step 2:\n', cmd)
proc = subprocess.run(cmd, shell=True, check=False)
print('Step 2 exit code:', proc.returncode)

## 8. Results

After both steps finish, the output files are at:
```
/kaggle/working/output/Extract-Classify-QUAD/<DOMAIN>_1st/   — Step 1 model + pred4pipeline.txt
/kaggle/working/output/Extract-Classify-QUAD/<DOMAIN>_2nd/   — Step 2 model + Test_results.txt + result.txt
```
Download them from the Kaggle output panel after the run completes.

In [ ]:
# Print the final test results from both steps
import os

for step_dir, label in [
    (os.path.join(OUTPUT_BASE, 'output', 'Extract-Classify-QUAD', DOMAIN + '_1st'), 'Step 1'),
    (os.path.join(OUTPUT_BASE, 'output', 'Extract-Classify-QUAD', DOMAIN + '_2nd'), 'Step 2'),
]:
    result_file = os.path.join(step_dir, 'Test_results.txt')
    if os.path.exists(result_file):
        print(f'\n=== {label} Test Results ===')
        with open(result_file, encoding='utf-8') as f:
            print(f.read())
    else:
        print(f'\n=== {label}: Test_results.txt not found at {result_file} ===')

## Notes

### Amharic-specific notes
- **Model**: `bert-base-multilingual-cased` supports Ethiopic/Ge'ez script. For best results, fine-tune on an Amharic-specific pre-trained model if available (e.g., AfriBERTa, AfroXLMR, or a dedicated Amharic BERT). Set the path in `BERT_MODEL`.
- **Data size**: The sample TSVs in `data/Amharic-ACOS/` contain only illustrative sentences. Replace them with a real annotated corpus (minimum ~500 sentences recommended for reasonable results).
- **Tokenization**: The pipeline uses whitespace tokenization for span alignment, consistent with the English domains. Amharic words should be space-delimited in your TSV files.
- **Categories**: The 31 Amharic aspect categories are defined in `run_classifier_dataset_utils.py`. Edit the `amharic` branch to match your annotation schema if needed.
- **Epochs**: More epochs are needed for a new language. Start with 10–20 for Step 1 and Step 2.

### General notes
- Enable GPU in Kaggle notebook settings for practical training speed.
- Download results from `/kaggle/working/output/Extract-Classify-QUAD/` after the run.